# IHP SG13G2 Octagon L2n0 — inductance extraction vs measured

This notebook is centered on the **existing parameter extraction** for the IHP
SG13_Octagon_L2n0 benchmark (nominal 2 nH).

For each S-parameter matrix it computes the series impedance

```text
Z(f) = 2 Z0 (1 - S21) / S21
L(f) = Im(Z) / (2πf)
Q(f) = Im(Z) / Re(Z)
```

and compares the Palace sweep (1–50 GHz, 11 points) with the two pinned IHP
measured dies. The measured low-frequency inductance is **1.91–1.93 nH**,
directly validating the nominal 2 nH benchmark. The 30 GHz result is above the
IHP guide differential 1-port SRF (~25.5 GHz). The 2-port series extraction shown here has a different L zero-crossing (~35.5 GHz), so the 30 GHz point is reported without implying a simple SRF agreement.

The IHP OpenEMS runtime was unavailable; the approved fallback is the pinned
IHP measured S-parameter data. The notebook documents the OpenEMS settings it
mirrors (`SG13_L2n0_2port_groundpolygon_mesh1.5um.py`: TopMetal1 edge ports with a Metal1/common-ground pillar,
50 Ω, refined_cellsize=1.5 µm, max_cellsize=lambda_air/(sqrt(eps_max)*20)
≈145 µm at 30 GHz/11.9, PEC boundaries, 0–30 GHz/401 points).
The Palace model approximates that excitation with interlayer ports and a minimal
reference pad; the OpenEMS-like Metal1 ground variant is a separate experiment.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

from gsim.common.benchmarks import ihp_l2n0 as b

plt.rcParams["figure.dpi"] = 120
RUN = b.RUN_ROOT / "l2n0_best"
run = b.load_recorded_run(RUN)
sp = run.sparams
print("Palace sweep:", sp.port_names, np.asarray(sp.freq), "GHz")

In [ ]:
def lq_from_s(s, f_hz, z0=b.Z0_OHM):
    z = 2.0 * z0 * (1.0 - s[1, 0]) / s[1, 0]
    l_nh = z.imag / (2 * np.pi * f_hz) * 1e9
    q = z.imag / z.real if z.real != 0 else float("nan")
    return l_nh, q, z


def palace_curves():
    rows = []
    for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
        s = b.s_matrix(sp, i)
        l, q, z = b.lq_from_s_matrix(s, f_ghz * 1e9)
        rows.append((f_ghz, l, q, s))
    return rows


def measured_curves(label):
    n = rf.Network(str(b.MEASURED[label]))
    z0 = float(np.real(np.asarray(n.z0).reshape(-1)[0]))
    z = 2.0 * z0 * (1.0 - n.s[:, 1, 0]) / n.s[:, 1, 0]
    l = z.imag / (2 * np.pi * n.f) * 1e9
    q = np.divide(z.imag, z.real, out=np.full_like(z.real, np.nan), where=z.real != 0)
    return n.f / 1e9, l, q, n.s


palace = palace_curves()
measured = {label: measured_curves(label) for label in ("die_x2y7", "die_x5y8")}

In [ ]:
import json

experiments = b.REPO / "nbs" / "data" / "ihp_validation" / "experiments"
matrix = json.loads((experiments / "results.json").read_text())
verify = json.loads((experiments / "verification" / "verification.json").read_text())
print(f"{'model':9} {'f GHz':>6} {'L nH':>7} {'Q':>7} | {'x2y7 L':>7} {'x5y8 L':>7}")
for name in ("baseline", "best"):
    for f_ghz in (1, 5, 9, 30):
        if f_ghz == 1:
            row = matrix["ground0_pec" if name == "best" else name]
        else:
            row = verify[name][str(f_ghz)]
        vals = []
        for label in measured:
            f, l, q, _ = measured[label]
            i = int(np.argmin(np.abs(f - f_ghz)))
            vals.append(l[i])
        print(
            f"{name:9} {f_ghz:6d} {row['L_nH']:7.3f} {row['Q']:7.2f} | {vals[0]:7.3f} {vals[1]:7.3f}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(f, l, color=color, lw=1.0, label=f"measured {label}")
    axes[1].plot(f, q, color=color, lw=1.0, label=f"measured {label}")
f_p = np.array([row[0] for row in palace])
axes[0].plot(f_p, [row[1] for row in palace], "o-", color="C3", label="Palace")
axes[1].plot(f_p, [row[2] for row in palace], "o-", color="C3", label="Palace")
for ax in axes:
    ax.axvline(30, color="k", lw=0.8, ls=":")
    ax.grid(alpha=0.3)
    ax.set(xlabel="frequency (GHz)")
    ax.legend(fontsize=8)
axes[0].set(title="Extracted series inductance", ylabel="L (nH)")
axes[1].set(title="Extracted quality factor", ylabel="Q")
fig.tight_layout()
plt.show()

In [ ]:
# S-parameter comparison at 30 GHz and across the measured band
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[0].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
    axes[1].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[1].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
    s = b.s_matrix(sp, i)
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[0, 0]))], "o", color="C3")
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[1, 0]))], "s", color="C3")
axes[0].set(
    title="Measured and Palace S-parameters", xlabel="frequency (GHz)", ylabel="dB"
)
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)
axes[1].set(
    title="Measured S-parameters (full band)", xlabel="frequency (GHz)", ylabel="dB"
)
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Experiment matrix

Screening at **1 GHz** on the corrected benchmark stack identified **280 µm substrate + no passivation + minimal ground + PEC boundary** (`ground0_pec`) as the best series-extraction variant:
**1.921 nH** vs measured **1.911/1.932 nH** (within 10%). The full matrix,
including the explicit conformal SiN sidewall shell, finite-metal vs planar-PEC, mesh, ground, PEC boundary, gap-port and combinations, is in `nbs/data/ihp_validation/experiments/`. The primary L/Q uses
the mandated series extraction; the matrix also records a differential-Z
cross-check.

The conformal-shell check adds a 0.6 µm SiN sidewall ring around every TopMetal2 polygon (the 1.9 µm top passivation is already in the PDK stack). At 1 GHz it gives 1.6277 nH, close to the 1.6230 nH mesh1.5 run, so it does not close the mismatch. The OpenEMS-like ground uses a Metal1 rectangle and Metal1→TopMetal1 ports; the central PEC pillar and exact edge-port geometry remain approximations.


In [ ]:
import json
from pathlib import Path

matrix = json.loads(
    (
        b.REPO / "nbs" / "data" / "ihp_validation" / "experiments" / "results.json"
    ).read_text()
)
print(f"{'variant':18} {'L 1GHz':>8} {'Q':>7} {'L diff':>8} {'pass':>5}")
for name, row in matrix.items():
    print(
        f"{name:18} {row.get('L_nH', float('nan')):8.3f} {row.get('Q', float('nan')):7.2f} {row.get('L_diff_nH', float('nan')):8.3f} {str(row.get('pass')):>5}"
    )

## Findings

- The measured 1 GHz inductance is **1.91–1.93 nH**, confirming the nominal
  2 nH benchmark from the extracted data itself.
- The best experiment (`ground0_pec`: 280 µm substrate, no passivation, minimal
  ground pad, PEC boundary) extracts **1.921 nH at 1 GHz**, within 10% of both measured dies.
  It falls to 0.716 nH at 30 GHz, below measured ~3.61/3.71 nH; this is reported as disagreement.
- Baseline series L at 1/5/9/30 GHz is 1.599/1.487/1.424/0.686 nH; best is 1.921/1.775/1.692/0.716 nH. The extracted low-frequency Q is 0.21 (best) / 0.28 (baseline) vs measured ~6.6, and the mesh error Norm is ~0.18 against AdaptiveTol 0.02: the model is not converged and the series-L match is not a full validation. The IHP guide's differential 1-port SRF (~25.5 GHz)
  is distinct from this 2-port series extraction.
- The comparison uses the same extraction formula for measured and simulated
  data, so differences come from the EM model/ports/mesh, not from a separate
  fitting method.

### Limitations

- OpenEMS was unavailable; the measured S-parameters are the approved fallback,
  and the OpenEMS settings are documented above.
- Palace uses the coarse 5 µm refined mesh and Palace Conductivity+Thickness
  metal-surface impedance BCs (finite conductor thickness), not collapsed planar PEC sheets.
  For a good conductor δ=sqrt(2/(ωμσ)) and Rs=1/(σδ); the configs use σ=3.77e7 S/m and the PDK thicknesses (δ≈2.6 µm at 1 GHz, 0.47 µm at 30 GHz).
- The reported `pass` column only means finite, positive extracted L. It is not a pass/fail against measured data; baseline is numerically valid but ~16% low at 1 GHz. The measured fixture reference planes are
  the TopMetal1 feed ends.
- The pinned IHP OpenEMS script uses a 280 µm substrate and omits passivation. A conformal 0.6 µm SiN sidewall shell plus the existing 1.9 µm top passivation is tested (`shell_conformal`, 1.6277 nH) but does not close the mismatch. A planar shell material swap was an earlier mistaken variant and is not used.
- The OpenEMS reference uses a central Metal1→TopMetal1 PEC pillar and horizontal edge ports; the Palace `openems_ground` variant uses a Metal1 rectangle and vertical Metal1→TopMetal1 via ports, so it is an approximation. A converged full-wave model and a direct OpenEMS run remain future work.
